In [1]:
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


In [2]:
print("Loading data...")
df = pd.read_csv('spam.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

encoder = LabelEncoder()
df['label_num'] = encoder.fit_transform(df['label'])

Loading data...


In [3]:
nltk.download('stopwords')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text) 
    
    tokens = text.split()
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

print("Cleaning text...")
df['clean_message'] = df['message'].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marou\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cleaning text...


In [4]:
print("Vectorizing text with TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 3), min_df=2, max_df=0.95)

X = vectorizer.fit_transform(df['clean_message'])
y = df['label_num']

Vectorizing text with TF-IDF...


In [5]:
print("Building Random Forest Model...")
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced', max_depth=None, max_features='sqrt', min_samples_leaf=1)


Building Random Forest Model...


In [6]:
print("Training...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df['label_num'])

rf_model.fit(X_train, y_train)

print("\nEvaluating model...")
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Not Spam', 'Spam']))

Training...

Evaluating model...
Test Accuracy: 97.40%

--- Classification Report ---
              precision    recall  f1-score   support

    Not Spam       0.97      1.00      0.99       966
        Spam       0.99      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [7]:
print("Saving model and vectorizer...")
with open('rf_spam_model.pkl', 'wb') as file:
    pickle.dump(rf_model, file)

with open('rf_vectorizer.pickle', 'wb') as file:
    pickle.dump(vectorizer, file, protocol=pickle.HIGHEST_PROTOCOL)

print("Training complete! Files saved: 'rf_spam_model.pkl' and 'rf_vectorizer.pickle'")


Saving model and vectorizer...
Training complete! Files saved: 'rf_spam_model.pkl' and 'rf_vectorizer.pickle'


In [8]:
def predict_spam_rf(input_text):
    """Function to be called by the GUI to get a prediction."""
    cleaned_text = clean_text(input_text)
    
    tfidf_matrix = vectorizer.transform([cleaned_text])
    
    prediction_num = rf_model.predict(tfidf_matrix)[0]
    
    if prediction_num == 1:
        return "Spam"
    else:
        return "Not Spam"

In [9]:
test_msg = "to reset your password, click the link below and follow the instructions"
print(f"\nTest Prediction for: '{test_msg}'")
print(f"Result: {predict_spam_rf(test_msg)}")


Test Prediction for: 'to reset your password, click the link below and follow the instructions'
Result: Not Spam


In [10]:
import os
print(os.getcwd())

c:\Cpyprojects\NLP\NLP-spam-detection
